# 🌿 Plant Disease Detection Model Training
1. Download PlantVillage dataset from Mendeley
2. Upload and Extract Dataset
This notebook trains a CNN model using MobileNetV2 transfer learning on the PlantVillage dataset (54,000+ images, 38 classes).

**Steps:**
1. Upload Kaggle API key
2. Download PlantVillage dataset
3. Train CNN model (2-phase: feature extraction + fine-tuning)
4. Convert to TensorFlow Lite
5. Download the `.tflite` model

> ⚠️ **Make sure GPU is enabled:** Runtime → Change runtime type → T4 GPU

## Step 1: Setup & Install Dependencies

In [ ]:
!pip install -q kaggle tensorflow matplotlib numpy

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

## Step 2: Upload PlantVillage Dataset

1. Download the full PlantVillage dataset (without augmentation) from: https://data.mendeley.com/datasets/tywbtsjrjv/1
2. Save the zip file, then upload it when prompted below

In [ ]:
from google.colab import files

# Upload the PlantVillage dataset zip file
print('Upload the dataset zip file you downloaded:')
uploaded = files.upload()

zip_filename = list(uploaded.keys())[0]
print(f'✅ Uploaded {zip_filename}!')

In [ ]:
# Extract PlantVillage dataset
!unzip -q "$zip_filename" -d /content/dataset

# Find the actual image directory
import glob
dirs = glob.glob('/content/dataset/**/color', recursive=True)
if dirs:
    DATASET_PATH = dirs[0]
else:
    # Try alternate structure
    dirs = glob.glob('/content/dataset/**/PlantVillage', recursive=True)
    if dirs:
        DATASET_PATH = dirs[0]
    else:
        DATASET_PATH = '/content/dataset'

print(f'Dataset path: {DATASET_PATH}')
classes = sorted(os.listdir(DATASET_PATH))
print(f'Number of classes: {len(classes)}')
print(f'Classes: {classes}')

## Step 3: Prepare Data

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Training data with augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

# Validation data - only rescaling
val_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

NUM_CLASSES = train_generator.num_classes
CLASS_NAMES = list(train_generator.class_indices.keys())

print(f'\nClasses: {NUM_CLASSES}')
print(f'Training samples: {train_generator.samples}')
print(f'Validation samples: {val_generator.samples}')

## Step 4: Build Model (MobileNetV2 Transfer Learning)

In [ ]:
# Create model with MobileNetV2 backbone
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model
base_model.trainable = False

model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 5: Train — Phase 1 (Feature Extraction)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7
    )
]

print('=== Phase 1: Feature Extraction (frozen base) ===')
history1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=callbacks,
    verbose=1
)

## Step 6: Train — Phase 2 (Fine-Tuning)

In [ ]:
# Unfreeze top layers of base model for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('=== Phase 2: Fine-Tuning ===')
history2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

## Step 7: Evaluate & Plot Results

In [ ]:
# Evaluate
loss, accuracy = model.evaluate(val_generator)
print(f'\n✅ Validation Accuracy: {accuracy * 100:.2f}%')
print(f'Validation Loss: {loss:.4f}')

# Plot training history
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss_hist = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(acc, label='Train'); ax1.plot(val_acc, label='Val')
ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(True)
ax2.plot(loss_hist, label='Train'); ax2.plot(val_loss, label='Val')
ax2.set_title('Loss'); ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()

## Step 8: Convert to TensorFlow Lite & Download

In [ ]:
# Save Keras model
model.save('plant_disease_model.h5')
print('Keras model saved.')

# Convert to TFLite with quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = 'plant_disease_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f'\n✅ TFLite model saved: {tflite_path}')
print(f'Model size: {len(tflite_model) / (1024*1024):.1f} MB')

# Save class labels
with open('plant_labels.txt', 'w') as f:
    for name in CLASS_NAMES:
        f.write(name + '\n')
print(f'Labels saved: {len(CLASS_NAMES)} classes')

In [ ]:
# Download the files to your computer
from google.colab import files

files.download('plant_disease_model.tflite')
files.download('plant_labels.txt')
files.download('plant_disease_model.h5')

print('\n🎉 Done! Copy plant_disease_model.tflite to:')
print('   app/src/main/assets/plant_disease_model.tflite')